In [4]:
import numpy as np
import pandas as pd
from scipy.special import gammaln


## 8.3.1の再現

In [40]:
# 表8.1のデータ
data = [
    # Women
    ["Women", "18-23", 26, 12, 7, 45],
    ["Women", "24-40", 9, 21, 15, 45],
    ["Women", ">40",   5, 14, 41, 60],

    # Men
    ["Men", "18-23", 40, 17, 8, 65],
    ["Men", "24-40", 17, 15, 12, 44],
    ["Men", ">40",   8, 15, 18, 41],
]

columns = [
    "Sex",
    "Age",
    "No_or_little_importance",
    "Important",
    "Very_important",
    "Total"
]

df = pd.DataFrame(data, columns=columns)

df_dummies = pd.get_dummies(df, columns = ["Sex", "Age"], dtype = int)

# 反応の基準カテゴリーは、「重要でない/あまり重要でない」と設定
# 説明変数の基準カテゴリーは、「女性」、「年齢18-23歳」を設定
X = df_dummies[["Sex_Men", "Age_24-40", "Age_>40"]].values
y = df_dummies[["Important", "Very_important"]].values

# n
n = df_dummies["Total"].values

# 切片を追加
X = np.column_stack([np.ones(X.shape[0]), X])

In [41]:
n

array([45, 45, 60, 65, 44, 41])

In [42]:
y

array([[12,  7],
       [21, 15],
       [14, 41],
       [17,  8],
       [15, 12],
       [15, 18]])

In [45]:
df_dummies["No_or_little_importance"]

0    26
1     9
2     5
3    40
4    17
5     8
Name: No_or_little_importance, dtype: int64

In [51]:
gammaln(n - y.sum(axis = 1) + 1)

array([ 61.26170176,  12.80182748,   4.78749174, 110.32063971,
        33.50507345,  10.6046029 ])

In [56]:
(gammaln(n + 1) - (gammaln(y + 1).sum(axis = 1) + gammaln(n - y.sum(axis = 1) + 1))).sum()

np.float64(264.9806754062721)

In [28]:
(gammaln(n + 1) - gammaln(y + 1).sum(axis = 1)).sum()

np.float64(498.26201245777725)

In [19]:
gammaln(n)

array([125.31727115, 125.31727115, 184.53382886, 205.16819948,
       121.53308152, 110.32063971])

np.float64(4.787491742782046)

In [26]:
gammaln(5)

np.float64(3.1780538303479458)

In [29]:
class MultinomialLogitOptimizer:
    
    def __init__(self, X, y, n):
        """
        X: 説明変数の行列 (N x (p+1))  ※0列目は1（切片）
        y: 各カテゴリの反応回数 (N x (J-1)) ※基準カテゴリを除いたもの
        n: 各共変量パターンの総試行回数 (N)
        """
        self.X = X
        self.y = y
        self.n = n
                
        # 係数ベクトル b の初期化 ( (p+1)*(J-1) 次元)
        self.beta = np.zeros((self.y.shape[1], self.X.shape[1]))

    # カテゴリー分類確率
    def pi_ij(self):
        
        # 線形予測子 z の計算
        z = self.X @ self.beta.T
        
        # 行ごとの最大値を差し引いてオーバーフローを防止
        # keepdims=True により (n_samples, 1) となり、ブロードキャストが可能
        z_max = np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z - z_max)
        
        # 分母の計算 (参照カテゴリの 1 を含める)
        # 1に対しても exp(-z_max) を掛けることで数学的整合性を保つ
        denominator = np.exp(-z_max) + exp_z.sum(axis=1, keepdims=True)
        
        return exp_z / denominator
    
    # スコアおよび情報行列の計算
    def compute_score_and_information_matrix(self):
                
        # スコアの計算
        U_lj = self.X.T @ (self.y - self.n.reshape(self.X.shape[0], 1) * self.pi_ij())    
        U = np.concatenate([U_lj[:, 0], U_lj[:, 1]])
    
        J = []
        for j in range(U_lj.shape[1]):
            for l in range(U_lj.shape[0]):
                for k in range(U_lj.shape[1]):
                    for s in range(U_lj.shape[0]):
                        if j == k:
                            w = (self.n * self.X[:, l] * self.X[:, s] * self.pi_ij()[:, j] * (1 - self.pi_ij()[:, j])).sum()
                        else:
                            w = (-1)*(self.n * self.pi_ij()[:, j] * self.pi_ij()[:, k] * self.X[:, l] * self.X[:, s]).sum()   
                        J.append(w)
    
        # 情報行列の計算
        J = np.array(J).reshape((U_lj.shape[0]*  U_lj.shape[1], U_lj.shape[0] *  U_lj.shape[1]))

        return U, J

    # IRLSによる更新
    def fit(self, n_iters = 10):

        for iter_ in range(n_iters):

            U, J = self.compute_score_and_information_matrix()
    
            # betaの更新
            beta_new = self.beta.flatten() + np.linalg.inv(J) @ U
            
            # 更新前後の誤差
            delta = ((beta_new - self.beta.flatten())**2).sum()
            self.beta = beta_new.reshape((y.shape[1], X.shape[1]))        
            
            # betaの更新幅
            if delta < 1e-10:
                break
            
        return self.beta, np.sqrt(np.diag(np.linalg.inv(J)))

In [30]:
cls = MultinomialLogitOptimizer(X, y, n)
beta, std_error = cls.fit()

In [31]:
std_error

array([0.28397548, 0.30051129, 0.3416449 , 0.40289909, 0.33050162,
       0.32103815, 0.40092566, 0.42292744])

In [32]:
beta

array([[-0.59079879, -0.38812814,  1.12826576,  1.58770717],
       [-1.03907572, -0.81301751,  1.47810691,  2.91675142]])

| 係数   | 推定値        | 標準誤差      |
|-------|---------------|---------------|
| β1    | -0.59079879   | 0.28397548    |
| β2    | -0.38812814   | 0.30051129    |
| β3    |  1.12826576   | 0.34164490    |
| β4    |  1.58770717   | 0.40289909    |
| β5    | -1.03907572   | 0.33050162    |
| β6    | -0.81301751   | 0.32103815    |
| β7    |  1.47810691   | 0.40092566    |
| β8    |  2.91675142   | 0.42292744    |


In [79]:
y

array([[12,  7],
       [21, 15],
       [14, 41],
       [17,  8],
       [15, 12],
       [15, 18]])

In [78]:
(y * np.log(cls.pi_ij())).sum() + ((n - y.sum(axis = 1)) * (1 - cls.pi_ij().sum(axis = 1))).sum()

np.float64(-150.3297792479579)

In [74]:
((n - y.sum(axis = 1)) * (1 - cls.pi_ij().sum(axis = 1))).sum()

np.float64(49.68852220724665)

In [80]:
1 - cls.pi_ij().sum(axis = 1)

array([0.52420072, 0.23458378, 0.09757829, 0.65247642, 0.35099386,
       0.17427567])

In [82]:
item1 = (y * np.log(cls.pi_ij())).sum()
item2 = ((n - y.sum(axis = 1)) * (1 - cls.pi_ij().sum(axis = 1))).sum()

In [83]:
cls.pi_ij()

array([[0.29034675, 0.18545253],
       [0.401529  , 0.36388722],
       [0.26442652, 0.63799519],
       [0.24514456, 0.10237902],
       [0.40752716, 0.24147898],
       [0.32035143, 0.5053729 ]])

In [85]:
item1

np.float64(-200.01830145520455)

In [86]:
item2

np.float64(49.68852220724665)

In [92]:
n

array([45, 45, 60, 65, 44, 41])

In [100]:
gammaln(n + 1)

array([129.12393364, 129.12393364, 188.62817342, 209.34258675,
       125.31727115, 114.03421178])

In [104]:
gammaln(n.reshape(X.shape[0], 1) + 1)

array([[129.12393364],
       [129.12393364],
       [188.62817342],
       [209.34258675],
       [125.31727115],
       [114.03421178]])

In [97]:
gammaln(y + 1)

array([[ 19.9872145 ,   8.52516136],
       [ 45.3801389 ,  27.89927138],
       [ 25.19122118, 114.03421178],
       [ 33.50507345,  10.6046029 ],
       [ 27.89927138,  19.9872145 ],
       [ 27.89927138,  36.39544521]])

In [112]:
(gammaln(n.flatten() + 1) 
- (
                gammaln(y + 1).sum(axis = 1) 
                + gammaln(n.flatten() - y.sum(axis = 1) + 1)
                )
).sum()

np.float64(264.9806754062721)

In [121]:
pi_min = y.sum(axis=0, keepdims=True) / n.sum()  # 各カテゴリの周辺確率
#pi_min = np.tile(pi_min, (y.shape[0], 1))

In [122]:
y.sum(axis=0, keepdims=True)

array([[ 94, 101]])

In [123]:
np.tile(pi_min, (y.shape[0], 1))

array([[0.31333333, 0.33666667],
       [0.31333333, 0.33666667],
       [0.31333333, 0.33666667],
       [0.31333333, 0.33666667],
       [0.31333333, 0.33666667],
       [0.31333333, 0.33666667]])

In [125]:
1 - 290.35 / 329.07

0.11766493451241367

In [126]:
1 - 25.37 / 64.29

0.6053818634313268